# Topic Modeling for the "Renaissance of General Relativity" Corpus

## Project Context

This notebook is part of the **"Trajectories of Change"** research project, which analyzes the evolution of knowledge systems using quantitative methods. The theoretical framework is based on **Socio-Epistemic Networks (SEN)**.

> Schlattmann, R. & Vogl, M. (2023). *Trajectories of Change: Approaches for Tracking Knowledge Evolution*. Big Historical Data Conference (BHDC), Jena, Germany.

## Notebook Purpose

1. **Document Embedding**: Convert scientific abstracts into vector representations
2. **Dimensionality Reduction**: Project embeddings (5D for clustering, 2D for visualization)
3. **Topic Clustering**: Identify thematic clusters using density-based clustering
4. **Topic Labeling**: Generate interpretable labels using LLMs

## Configurable Providers

This notebook supports **3 providers** for both embeddings and LLM labeling:

| Provider | Embeddings | LLM Labeling | Notes |
|----------|------------|--------------|-------|
| `huggingface_api` | HF Inference API | HF Inference API | Remote, per-token cost |
| `local` | SentenceTransformers | Transformers | No API cost, requires GPU |
| `openrouter` | OpenRouter API | OpenRouter API | Central billing, many models |

## Key Configuration Options

| Setting | Options | Default |
|---------|---------|---------|
| `EMBEDDING_PROVIDER` | `'huggingface_api'`, `'local'`, `'openrouter'` | `'openrouter'` |
| `LLM_PROVIDER` | `'huggingface_api'`, `'local'`, `'openrouter'` | `'openrouter'` |
| `PIPELINE_MODELS` | `['MMR']`, `['KeyBERT']`, `[]` | `['MMR']` |
| `PARALLEL_MODELS` | `['MMR', 'POS', 'KeyBERT']` | `['MMR', 'POS']` |
| `SAMPLE_SIZE` | `None` or integer | `None` (all data) |
| `DIM_REDUCTION` | `'umap'`, `'pacmap'` | `'pacmap'` |
| `CLUSTERING` | `'hdbscan'`, `'fast_hdbscan'` | `'fast_hdbscan'` |

## Output Files

- `df_sampled_full_{provider}_{model}_{dim_reduction}_{clustering}.parquet`
- `bertopic_model_full_{provider}_{model}_{dim_reduction}_{clustering}/`

## Two Projections

| Projection | Dims | Purpose |
|------------|------|---------|
| 5D | 5 | Clustering input |
| 2D | 2 | Visualization & KDE |

In [1]:
# =============================================================================
# IMPORTS
# =============================================================================
# Standard library
import os
import sys
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# Data & ML
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer
from dotenv import load_dotenv

# Embeddings & Topic Modeling
import litellm
import openai
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.dimensionality import BaseDimensionalityReduction
from bertopic.representation import MaximalMarginalRelevance, PartOfSpeech, LiteLLM

# Optional imports (availability checked after config)
try:
    from transformers import pipeline as hf_pipeline
    from bertopic.representation import TextGeneration
except ImportError:
    hf_pipeline = None
    TextGeneration = None

try:
    from umap import UMAP
except ImportError:
    UMAP = None

try:
    import pacmap
except ImportError:
    pacmap = None

try:
    from hdbscan import HDBSCAN
except ImportError:
    HDBSCAN = None

try:
    import fast_hdbscan
except ImportError:
    fast_hdbscan = None

try:
    from bertopic.representation import KeyBERTInspired
except ImportError:
    KeyBERTInspired = None

# NLP
import spacy
try:
    spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")

print("All imports loaded successfully.")

All imports loaded successfully.


In [2]:
# =============================================================================
# CONFIGURATION - Topic Modeling Pipeline
# =============================================================================
load_dotenv()

# -----------------------------------------------------------------------------
# Paths (imported from central config)
# -----------------------------------------------------------------------------
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from config import (PROJECT_ROOT, DATA_DIR, EMBEDDINGS_CACHE, DIM_REDUCTION_CACHE, 
                    MODELS_DIR, INPUT_JSON, PLOTS_DIR)

INPUT_FILE = INPUT_JSON

# =============================================================================
# SAMPLING (for testing - None = all data, integer = sample size)
# =============================================================================
SAMPLE_SIZE = 200  # e.g., 200 for quick tests, None for production

# =============================================================================
# EMBEDDING PROVIDER
# Options: 'huggingface_api', 'local', 'openrouter'
# =============================================================================
EMBEDDING_PROVIDER = 'local'

# Provider-specific models:
# - huggingface_api: 'huggingface/BAAI/bge-large-en-v1.5', 'huggingface/sentence-transformers/all-MiniLM-L6-v2'
# - local:           'BAAI/bge-large-en-v1.5', 'sentence-transformers/all-MiniLM-L6-v2', 'intfloat/multilingual-e5-large'
# - openrouter:      'openai/text-embedding-3-large', 'openai/text-embedding-3-small', 'google/gemini-embedding-001', 'qwen/qwen3-embedding-8b'
EMBEDDING_MODEL = 'google/embeddinggemma-300m'

# =============================================================================
# LLM LABELING PROVIDER
# Options: 'huggingface_api', 'local', 'openrouter'
# =============================================================================
LLM_PROVIDER = 'local'

# Provider-specific models:
# - huggingface_api: 'huggingface/mistralai/Mixtral-8x7B-Instruct-v0.1', 'huggingface/meta-llama/Llama-2-70b-chat-hf'
# - local:           'mistralai/Mistral-7B-Instruct-v0.2', 'meta-llama/Llama-2-7b-chat-hf'
# - openrouter:      'google/gemini-2.5-flash-preview-09-2025', 'anthropic/claude-3-5-sonnet', 'openai/gpt-4o-mini'
LLM_MODEL = 'google/gemma-3-1b-it'

# =============================================================================
# OPENROUTER-SPECIFIC: Provider Routing
# (Only relevant when EMBEDDING_PROVIDER='openrouter' or LLM_PROVIDER='openrouter')
# =============================================================================
OPENROUTER_PROVIDER_ROUTING = {
    "sort": "latency"  # Fastest provider
}
OPENROUTER_API_BASE = "https://openrouter.ai/api/v1"

# Make LiteLLM point to OpenRouter globally
litellm.api_base = os.getenv("OPENROUTER_API_BASE", OPENROUTER_API_BASE)
litellm.api_key = os.getenv("OPENROUTER_API_KEY")

# =============================================================================
# LLM REPRESENTATION PARAMETERS
# =============================================================================
LLM_NR_DOCS = 8                   # Number of representative documents per topic
LLM_DIVERSITY = 0.2               # Document diversity (0-1), 0.2 for scientific texts
LLM_DELAY = 0.3                   # Delay between API calls (rate limiting)
# Note: doc_length and tokenizer are NOT supported by BERTopic's LiteLLM class

LLM_PROMPT = """You are an experienced researcher in gravitational physics, astrophysics, and cosmology. You are labeling research topic clusters based on scientific abstracts.

Documents: [DOCUMENTS]
Keywords: [KEYWORDS]

Task:
- Generate EXACTLY ONE topic label.
- The label MUST contain between 4 and 7 words (inclusive).
- The label should read like a review article or textbook chapter title.

Guidelines:
- Use standard physics terminology (e.g., "Kerr black hole perturbations", "post-Newtonian approximation methods").
- Be specific about the physical phenomenon or method.
- Avoid generic terms like "studies", "analysis", or "research".

Output format (single line):
topic: <label>

Do NOT write anything else (no explanations, no additional sentences, no quotes, no bullet points)."""

# =============================================================================
# REPRESENTATION MODEL CONFIGURATION
# =============================================================================
# MMR diversity for keyword selection (0-1, higher = more diverse)
MMR_DIVERSITY = 0.3

# Pipeline models run BEFORE LLM (sequentially, to diversify keywords for LLM)
# Options: 'POS', 'KeyBERT', 'MMR', or [] for LLM only
PIPELINE_MODELS = ['POS', 'KeyBERT', 'MMR']

# Parallel models stored SEPARATELY in topic_aspects_ (for comparison/analysis)
# Options: 'MMR', 'KeyBERT', 'POS'
PARALLEL_MODELS = ['MMR', 'POS', 'KeyBERT']

# KeyBERTInspired requires a LOCAL embedding model (even when using API for documents)
# Options:
#   'sentence-transformers/all-MiniLM-L6-v2'    - Fast (80MB), good quality
#   'sentence-transformers/all-mpnet-base-v2'  - Balanced (420MB), better quality
#   'BAAI/bge-large-en-v1.5'                   - Best for scientific texts (1.3GB)
KEYBERT_EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

# =============================================================================
# COST TRACKING
# =============================================================================
TRACK_COSTS = True

# Token and cost counters (accumulated during execution)
total_embedding_tokens = 0
total_llm_input_tokens = 0
total_llm_output_tokens = 0
total_embedding_cost = 0.0
total_llm_cost = 0.0

# LiteLLM callback for tracking usage across embeddings + chat completions
# Relies on LiteLLM's response_cost and usage fields (no manual pricing)
def _track_tokens_and_cost(kwargs, completion_response, start_time, end_time):
    global total_embedding_tokens, total_llm_input_tokens, total_llm_output_tokens
    global total_embedding_cost, total_llm_cost

    usage = getattr(completion_response, 'usage', None)
    if usage is None and isinstance(completion_response, dict):
        usage = completion_response.get('usage')
    usage = usage or {}

    call_type = kwargs.get('call_type')
    if call_type is None:
        call_type = 'embedding' if kwargs.get('input') and not kwargs.get('messages') else 'completion'

    prompt_tokens = usage.get('prompt_tokens') or usage.get('input_tokens') or 0
    completion_tokens = usage.get('completion_tokens') or usage.get('output_tokens') or 0
    total_tokens = usage.get('total_tokens') or (prompt_tokens + completion_tokens)

    cost = kwargs.get('response_cost')
    if cost is None:
        cost = getattr(completion_response, '_hidden_params', {}).get('response_cost') if not isinstance(completion_response, dict) else completion_response.get('_hidden_params', {}).get('response_cost')
    if cost is None:
        cost = usage.get('cost') or 0.0

    if call_type == 'embedding':
        total_embedding_tokens += total_tokens
        total_embedding_cost += float(cost)
    else:
        total_llm_input_tokens += prompt_tokens
        total_llm_output_tokens += completion_tokens
        total_llm_cost += float(cost)

# Register the callback globally
litellm.success_callback = [_track_tokens_and_cost]

# =============================================================================
# MODEL SELECTION - Dimensionality Reduction & Clustering
# =============================================================================
DIM_REDUCTION = 'pacmap'          # 'umap' or 'pacmap'
CLUSTERING = 'fast_hdbscan'       # 'hdbscan' or 'fast_hdbscan'

# Display warnings for trade-offs
if DIM_REDUCTION == 'pacmap':
    print("WARNING: PaCMAP does not support densmap (density-preserving mode).")
    print("         KDE analysis in Notebook 04 may be affected.")
    print("         UMAP with densmap=True preserves local density better for KDE.")

if 'KeyBERT' in PIPELINE_MODELS or 'KeyBERT' in PARALLEL_MODELS:
    print("WARNING: KeyBERTInspired will compute NEW embeddings for keywords.")
    print(f"         Using model: {KEYBERT_EMBEDDING_MODEL}")

# =============================================================================
# EMBEDDING PARAMETERS
# =============================================================================
MAX_TOKENS = 8191                 # Max tokens per document (for truncation)
BATCH_SIZE = 64                   # Batch size for API calls
MAX_WORKERS = 5                   # Parallel workers for API calls
CACHE_DTYPE = np.float32          # Data type for cache

# =============================================================================
# DIMENSIONALITY REDUCTION CONFIG
# =============================================================================
RANDOM_STATE = 42

# UMAP-specific parameters
UMAP_N_NEIGHBORS = 80
UMAP_MIN_DIST = 0.05

UMAP_5D = dict(
    n_neighbors=UMAP_N_NEIGHBORS,
    n_components=5,
    min_dist=UMAP_MIN_DIST,
    metric='cosine',
    random_state=RANDOM_STATE,
    verbose=True
)

UMAP_2D = dict(
    n_neighbors=UMAP_N_NEIGHBORS,  
    n_components=2,
    min_dist=UMAP_MIN_DIST,        
    metric='cosine',
    densmap=True,
    random_state=RANDOM_STATE,
    verbose=True
)

# PaCMAP parameters
PACMAP_N_NEIGHBORS = 60
PACMAP_MN_RATIO = 0.5
PACMAP_FP_RATIO = 1.0

PACMAP_5D = dict(
    n_components=5,
    n_neighbors=PACMAP_N_NEIGHBORS,
    MN_ratio=PACMAP_MN_RATIO,
    FP_ratio=PACMAP_FP_RATIO,
    random_state=RANDOM_STATE,
    verbose=True
)

PACMAP_2D = dict(
    n_components=2,
    n_neighbors=PACMAP_N_NEIGHBORS,  
    MN_ratio=PACMAP_MN_RATIO,        
    FP_ratio=PACMAP_FP_RATIO,        
    random_state=RANDOM_STATE,
    verbose=True
)

# =============================================================================
# CLUSTERING CONFIG
# =============================================================================
# Base parameters for full dataset (~180k documents)
_BASE_MIN_CLUSTER_SIZE = 180  # ~0.2% of full data → 65-70 topics

# Dynamic adjustment for sampling: ensure we can find clusters
if SAMPLE_SIZE is not None:
    # Rule: min_cluster_size = ~5% of sample, minimum 10
    _MIN_CLUSTER_SIZE = max(10, int(SAMPLE_SIZE * 0.05))
    print(f"SAMPLING MODE: Adjusted min_cluster_size from {_BASE_MIN_CLUSTER_SIZE} to {_MIN_CLUSTER_SIZE}")
else:
    _MIN_CLUSTER_SIZE = _BASE_MIN_CLUSTER_SIZE

HDBSCAN_PARAMS = dict(
    min_cluster_size=_MIN_CLUSTER_SIZE,
    min_samples=3,
    metric='euclidean',
    cluster_selection_method='eom',
    cluster_selection_epsilon=0.02,
    prediction_data=True,
    gen_min_span_tree=True
)

FAST_HDBSCAN_PARAMS = dict(
    min_cluster_size=_MIN_CLUSTER_SIZE,
    min_samples=3,
    cluster_selection_method='eom',
    cluster_selection_epsilon=0.02,
    allow_single_cluster=False,
)

# =============================================================================
# OUTPUT FILES (derived from config)
# =============================================================================
# Cache filenames include provider + model for uniqueness
_embedding_model_safe = EMBEDDING_MODEL.replace('/', '_')
MODEL_SUFFIX = f"{EMBEDDING_PROVIDER}_{_embedding_model_safe}_{DIM_REDUCTION}_{CLUSTERING}"
EMBEDDINGS_FILE = EMBEDDINGS_CACHE / f"embeddings_{EMBEDDING_PROVIDER}_{_embedding_model_safe}.npz"
REDUCED_5D_FILE = DIM_REDUCTION_CACHE / f"reduced_5d_{MODEL_SUFFIX}.npy"
REDUCED_2D_FILE = DIM_REDUCTION_CACHE / f"reduced_2d_{MODEL_SUFFIX}.npy"
OUTPUT_PARQUET = DATA_DIR / f"df_sampled_full_{MODEL_SUFFIX}.parquet"
OUTPUT_MODEL = MODELS_DIR / f"bertopic_model_full_{MODEL_SUFFIX}"

# Backward compatibility: Recognize legacy caches without provider prefix
_legacy_cache = EMBEDDINGS_CACHE / f"embeddings_{_embedding_model_safe}.npz"
if _legacy_cache.exists() and not EMBEDDINGS_FILE.exists():
    print(f"INFO: Legacy cache found: {_legacy_cache.name}")
    print(f"      Using it (provider prefix missing in old format).")
    EMBEDDINGS_FILE = _legacy_cache

# =============================================================================
# CONFIGURATION SUMMARY
# =============================================================================
print(f"{'='*60}")
print(f"Configuration Summary")
print(f"{'='*60}")
print(f"Sampling:            {SAMPLE_SIZE if SAMPLE_SIZE else 'ALL DATA'}")
print(f"Embedding Provider:  {EMBEDDING_PROVIDER}")
print(f"Embedding Model:     {EMBEDDING_MODEL}")
print(f"LLM Provider:        {LLM_PROVIDER}")
print(f"LLM Model:           {LLM_MODEL}")
print(f"Pipeline Models:     {PIPELINE_MODELS if PIPELINE_MODELS else 'None'}")
print(f"Parallel Models:     {PARALLEL_MODELS if PARALLEL_MODELS else 'None'}")
print(f"Cost Tracking:       {'ENABLED' if TRACK_COSTS else 'DISABLED'}")
print(f"Dim. Reduction:      {DIM_REDUCTION.upper()}")
print(f"Clustering:          {CLUSTERING.upper()}")
print(f"Output:              {OUTPUT_PARQUET.name}")


         KDE analysis in Notebook 04 may be affected.
         UMAP with densmap=True preserves local density better for KDE.
         Using model: sentence-transformers/all-MiniLM-L6-v2
SAMPLING MODE: Adjusted min_cluster_size from 180 to 10
Configuration Summary
Sampling:            200
Embedding Provider:  local
Embedding Model:     google/embeddinggemma-300m
LLM Provider:        local
LLM Model:           google/gemma-3-1b-it
Pipeline Models:     ['POS', 'KeyBERT', 'MMR']
Parallel Models:     ['MMR', 'POS', 'KeyBERT']
Cost Tracking:       ENABLED
Dim. Reduction:      PACMAP
Clustering:          FAST_HDBSCAN
Output:              df_sampled_full_local_google_embeddinggemma-300m_pacmap_fast_hdbscan.parquet


In [3]:
# =============================================================================
# VALIDATE IMPORTS BASED ON CONFIG
# =============================================================================
# Check that required packages are available for selected providers

if EMBEDDING_PROVIDER == 'openrouter' and openai is None:
    raise ImportError("openai package required for EMBEDDING_PROVIDER='openrouter'. Install with: pip install openai")

if LLM_PROVIDER == 'local' and (hf_pipeline is None or TextGeneration is None):
    raise ImportError("transformers package required for LLM_PROVIDER='local'. Install with: pip install transformers")

if DIM_REDUCTION == 'umap' and UMAP is None:
    raise ImportError("umap-learn package required for DIM_REDUCTION='umap'. Install with: pip install umap-learn")

if DIM_REDUCTION == 'pacmap' and pacmap is None:
    raise ImportError("pacmap package required for DIM_REDUCTION='pacmap'. Install with: pip install pacmap")

if CLUSTERING == 'hdbscan' and HDBSCAN is None:
    raise ImportError("hdbscan package required for CLUSTERING='hdbscan'. Install with: pip install hdbscan")

if CLUSTERING == 'fast_hdbscan' and fast_hdbscan is None:
    raise ImportError("fast_hdbscan package required for CLUSTERING='fast_hdbscan'. Install with: pip install fast_hdbscan")

if ('KeyBERT' in PIPELINE_MODELS or 'KeyBERT' in PARALLEL_MODELS) and KeyBERTInspired is None:
    raise ImportError("KeyBERTInspired not available. Check BERTopic installation.")

print("All required imports validated successfully.")

All required imports validated successfully.


## Data: NASA ADS Corpus

**Source**: NASA Astrophysics Data System (ADS)

**Domain**: General Relativity and Gravitation Research (1911-2000)

| Property | Value |
|----------|-------|
| Documents | ~180,000 |
| Types | Journal articles, books, conference papers |
| Language | English (translated via OpenAI) |

**Text Fields**:
- `full_text`: Title + Abstract (used for embedding)
- `tokens`: Pre-processed token list (lemmatized, stopwords removed)

In [4]:
df = pd.read_json(INPUT_FILE, lines=True)

# Optional: Sampling for quick tests
if SAMPLE_SIZE is not None:
    df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"SAMPLING ACTIVE: {len(df):,} documents (requested {SAMPLE_SIZE:,})")
else:
    print(f"Loaded {len(df):,} documents (full dataset)")

documents = df['full_text'].tolist()
df.head(3)

SAMPLING ACTIVE: 200 documents (requested 200)


,Bibcode,Author,Title,Year,Journal,Journal Abbreviation,Issue,Volume,First Page,Last Page,...,Category,Citation Count,References,PDF_URL,Title_lang,Abstract_lang,Title_en,Abstract_en,full_text,tokens
0,2000IJNMF..32..263W,"Winters KB, Seim HE, Finnigan TD","Simulation of non-hydrostatic, density-stratif...",2000,International Journal for Numerical Methods in...,IJNMF,3,32,263,284,...,article,9.0,"[1983JCoPh..50...71R, 1983JPO....13..695H, 198...",https://ui.adsabs.harvard.edu/link_gateway/200...,en,en,"Simulation of non-hydrostatic, density-stratif...",A numerical model has been developed for simul...,"Simulation of non-hydrostatic, density-stratif...","[simulation, non, hydrostatic, density, strati..."
1,1980RSPTA.296..367W,Wall JV,Evidence for the Cosmological Evolution of Act...,1980,Philosophical Transactions of the Royal Societ...,RSPTA,1419,296,367,383,...,article,27.0,[],None,en,en,Evidence for the Cosmological Evolution of Act...,Investigations of the radial distribution of q...,Evidence for the Cosmological Evolution of Act...,"[evidence, cosmological, evolution, active, ga..."
2,1994PhRvE..49.4466M,Miyata K,Equalization of longitudinal and transverse be...,1994,Physical Review E,PhRvE,5,49,4466,4473,...,article,1.0,"[1958PhRv..111..373R, 1979ITNS...26.3604S, 198...",None,en,en,Equalization of longitudinal and transverse be...,This paper shows that longitudinal and transve...,Equalization of longitudinal and transverse be...,"[equalization, longitudinal, transverse, beam,..."


## Embedding Pipeline

This notebook supports **3 embedding providers** configurable in the CONFIG cell:

| Provider | Model Examples | Cost | Notes |
|----------|----------------|------|-------|
| `huggingface_api` | `huggingface/BAAI/bge-large-en-v1.5` | Per Token | HuggingFace Inference API |
| `local` | `BAAI/bge-large-en-v1.5` | None | Local SentenceTransformers |
| `openrouter` | `openai/text-embedding-3-large` | Per Token | Central billing via OpenRouter |

**Caching**: Embeddings are cached to `cache/embeddings/` with provider-specific filenames for easy re-runs.

**Cost Tracking**: When `TRACK_COSTS=True`, API costs are tracked and displayed at the end.

In [5]:
# =============================================================================
# EMBEDDING FUNCTIONS - 3 Provider Options
# =============================================================================

def get_embeddings_huggingface_api(documents, model_name, batch_size):
    """Get embeddings using HuggingFace Inference API via LiteLLM."""
    global total_embedding_tokens, total_embedding_cost
    all_embeddings = []
    batches = [documents[i:i + batch_size] for i in range(0, len(documents), batch_size)]

    for batch in tqdm(batches, desc=f"Embedding ({EMBEDDING_PROVIDER})"):
        for attempt in range(3):
            try:
                response = litellm.embedding(model=model_name, input=batch)
                batch_embeddings = [d['embedding'] for d in response['data']]
                all_embeddings.extend(batch_embeddings)
                break
            except Exception as e:
                if attempt == 2:
                    raise e
                time.sleep(1 * (attempt + 1))

    if TRACK_COSTS:
        print(f"Embedding tokens (HuggingFace via LiteLLM): {total_embedding_tokens:,}")
        print(f"Embedding cost (HuggingFace via LiteLLM): ${total_embedding_cost:.6f}")
    return np.array(all_embeddings, dtype=CACHE_DTYPE)


def get_embeddings_local(documents, model_name):
    """Get embeddings using local SentenceTransformers model."""
    print(f"Loading local model: {model_name}")
    model = SentenceTransformer(model_name)
    embeddings = model.encode(documents, show_progress_bar=True, batch_size=BATCH_SIZE)
    print(f"Embedding cost (local): $0.00")
    return embeddings.astype(CACHE_DTYPE)


def get_embeddings_openrouter(documents, model_name, batch_size):
    """Get embeddings using OpenRouter API via LiteLLM (OpenAI-compatible route)."""
    global total_embedding_tokens, total_embedding_cost

    all_embeddings = []
    batches = [documents[i:i + batch_size] for i in range(0, len(documents), batch_size)]

    # Normalize model: strip openrouter/, then prefix openai/ for OpenAI-compatible routing
    _base = model_name.split('openrouter/', 1)[1] if model_name.startswith('openrouter/') else model_name
    model = _base if _base.startswith('openai/') else f'openai/{_base}'

    for batch in tqdm(batches, desc=f"Embedding ({EMBEDDING_PROVIDER})"):
        for attempt in range(3):
            try:
                response = litellm.embedding(
                    model=model,
                    input=batch,
                    api_base=OPENROUTER_API_BASE,
                    metadata={"tags": ["bertopic", "embeddings"]},
                    extra_body={"provider": OPENROUTER_PROVIDER_ROUTING},
                )
                batch_embeddings = [d['embedding'] for d in response['data']]
                all_embeddings.extend(batch_embeddings)

                if TRACK_COSTS and len(all_embeddings) == len(batch):
                    usage = getattr(response, 'usage', None)
                    if usage is None and isinstance(response, dict):
                        usage = response.get('usage')
                    if not usage:
                        print("  ⚠ No usage data in response (provider may omit it)")
                break
            except Exception as e:
                if attempt == 2:
                    raise e
                time.sleep(1 * (attempt + 1))

    if TRACK_COSTS:
        print(f"Embedding tokens (OpenRouter/LiteLLM): {total_embedding_tokens:,}")
        print(f"Embedding cost   (OpenRouter/LiteLLM): ${total_embedding_cost:.6f}")
    return np.array(all_embeddings, dtype=CACHE_DTYPE)

def get_embeddings_by_provider(documents):
    """Route to correct embedding provider based on config."""
    if EMBEDDING_PROVIDER == 'huggingface_api':
        return get_embeddings_huggingface_api(documents, EMBEDDING_MODEL, BATCH_SIZE)
    elif EMBEDDING_PROVIDER == 'local':
        return get_embeddings_local(documents, EMBEDDING_MODEL)
    elif EMBEDDING_PROVIDER == 'openrouter':
        return get_embeddings_openrouter(documents, EMBEDDING_MODEL, BATCH_SIZE)
    else:
        raise ValueError(f"Unknown EMBEDDING_PROVIDER: {EMBEDDING_PROVIDER}")


def get_or_load_embeddings(documents):
    """Load cached embeddings or compute new ones."""
    if EMBEDDINGS_FILE.exists():
        data = np.load(EMBEDDINGS_FILE, allow_pickle=True)
        print(f"Loaded embeddings from cache: {EMBEDDINGS_FILE.name}")
        if TRACK_COSTS:
            print(f"Embedding cost: $0.00 (from cache)")
        return data['embeddings']

    print(f"Computing embeddings with {EMBEDDING_PROVIDER}/{EMBEDDING_MODEL}...")
    embeddings = get_embeddings_by_provider(documents)

    np.savez_compressed(EMBEDDINGS_FILE, embeddings=embeddings, model=EMBEDDING_MODEL, provider=EMBEDDING_PROVIDER)
    print(f"Saved embeddings to: {EMBEDDINGS_FILE.name}")
    return embeddings


embeddings = get_or_load_embeddings(documents)
print(f"Embeddings shape: {embeddings.shape}")

Loaded embeddings from cache: embeddings_local_google_embeddinggemma-300m.npz
Embedding cost: $0.00 (from cache)
Embeddings shape: (200, 768)


## Dimensionality Reduction

Two separate projections with caching, using the algorithm selected in config (`DIM_REDUCTION`):

| Projection | Purpose | Key Parameters |
|------------|---------|----------------|
| **5D** | HDBSCAN input | `n_neighbors=80` |
| **2D** | Visualization & KDE | `n_neighbors=150` |

**Note**: UMAP's `densmap=True` preserves density information for KDE analysis (Notebook 04). PaCMAP does not have this feature.

In [6]:
def create_dim_reduction_model(n_components):
    """Create dimensionality reduction model based on config."""
    if DIM_REDUCTION == 'umap':
        params = UMAP_5D.copy() if n_components == 5 else UMAP_2D.copy()
        return UMAP(**params)
    elif DIM_REDUCTION == 'pacmap':
        params = PACMAP_5D.copy() if n_components == 5 else PACMAP_2D.copy()
        return pacmap.PaCMAP(**params)
    else:
        raise ValueError(f"Unknown dimensionality reduction: {DIM_REDUCTION}")

def get_or_compute_reduction(embeddings, n_components, cache_file, name):
    """Load reduced embeddings from cache or compute and save."""
    if cache_file.exists():
        print(f"Loaded {name} from cache: {cache_file.name}")
        return np.load(cache_file)
    
    print(f"Computing {name} with {DIM_REDUCTION.upper()}... (this may take a while)")
    model = create_dim_reduction_model(n_components)
    reduced = model.fit_transform(embeddings)
    np.save(cache_file, reduced)
    print(f"Saved {name} to: {cache_file.name}")
    return reduced

# Compute BOTH 5D and 2D with the SAME algorithm for consistency
reduced_5d = get_or_compute_reduction(embeddings, 5, REDUCED_5D_FILE, "5D Reduction")
reduced_2d = get_or_compute_reduction(embeddings, 2, REDUCED_2D_FILE, "2D Reduction")

print(f"\n5D shape: {reduced_5d.shape}, 2D shape: {reduced_2d.shape}")

Loaded 5D Reduction from cache: reduced_5d_local_google_embeddinggemma-300m_pacmap_fast_hdbscan.npy
Loaded 2D Reduction from cache: reduced_2d_local_google_embeddinggemma-300m_pacmap_fast_hdbscan.npy

5D shape: (200, 5), 2D shape: (200, 2)


## Clustering (HDBSCAN)

HDBSCAN identifies topic clusters based on density in the 5D UMAP space:
- No pre-specified number of clusters
- Handles varying densities
- Outliers labeled as topic -1

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `min_cluster_size` | 350 | ~0.2% of data → ~65-70 topics |
| `cluster_selection_method` | eom | Selects most persistent clusters |

In [7]:
def create_clustering_model():
    """Create clustering model based on config."""
    if CLUSTERING == 'hdbscan':
        return HDBSCAN(**HDBSCAN_PARAMS)
    elif CLUSTERING == 'fast_hdbscan':
        return fast_hdbscan.HDBSCAN(**FAST_HDBSCAN_PARAMS)
    else:
        raise ValueError(f"Unknown clustering algorithm: {CLUSTERING}")

clustering_model = create_clustering_model()
clusters = clustering_model.fit_predict(reduced_5d)

n_topics = len(set(clusters)) - (1 if -1 in clusters else 0)
n_outliers = (clusters == -1).sum()
print(f"Using {CLUSTERING.upper()}")
print(f"Found {n_topics} topics, {n_outliers:,} outliers ({n_outliers/len(clusters)*100:.1f}%)")

Using FAST_HDBSCAN
Found 2 topics, 1 outliers (0.5%)


## Topic Labeling with BERTopic

This notebook supports **3 LLM providers** for topic labeling:

| Provider | Model Examples | Notes |
|----------|----------------|-------|
| `huggingface_api` | `huggingface/mistralai/Mixtral-8x7B-Instruct-v0.1` | HuggingFace Inference API |
| `local` | `mistralai/Mistral-7B-Instruct-v0.2` | Local Transformers model |
| `openrouter` | `openrouter/google/gemini-2.5-flash-preview-09-2025` | Central billing via OpenRouter |

### Hybrid Representation Model (Dictionary + Pipeline)

The representation model is configured as a **Dictionary** to enable both:
1. **Pipeline models** (`PIPELINE_MODELS`): Run sequentially BEFORE the LLM (e.g., MMR → LLM)
2. **Parallel models** (`PARALLEL_MODELS`): Run independently and stored separately in `topic_aspects_`

**Important**: The "Main" key (pipeline result) goes to `topic_representations_`, NOT `topic_aspects_`. Only parallel models appear in `topic_aspects_`.

**Available Models**:
| Model | Description | Use Case |
|-------|-------------|----------|
| `MMR` | Maximal Marginal Relevance | Diversify keywords (reduce redundancy) |
| `KeyBERT` | KeyBERTInspired re-ranking | Better keyword extraction |
| `POS` | Part-of-Speech filtering | Grammatical keyword selection |

Default (quality-first) pipeline: `POS → KeyBERT → MMR → LLM`. This keeps only grammatical phrases, re-ranks them semantically, diversifies, then hands the cleaned list to the LLM.

### KeyBERTInspired Embedding Models

KeyBERTInspired requires a **local SentenceTransformer** model (set via `KEYBERT_EMBEDDING_MODEL`):

| Model | Size | Speed | Quality | Use Case |
|-------|------|-------|---------|----------|
| `sentence-transformers/all-MiniLM-L6-v2` | 80MB | Fast | Good | Quick tests |
| `sentence-transformers/all-mpnet-base-v2` | 420MB | Medium | Better | Balanced |
| `BAAI/bge-large-en-v1.5` | 1.3GB | Slow | Best | Scientific texts |

### Parameters (CONFIG cell)
- `LLM_NR_DOCS`: Number of representative documents per topic (default: 8)
- `LLM_DIVERSITY`: Document diversity (0-1, default: 0.2 for scientific texts)
- `MMR_DIVERSITY`: Keyword diversity (0-1, default: 0.3)
- `PIPELINE_MODELS`: Models to run before LLM (default: `['MMR']`)
- `PARALLEL_MODELS`: Models to store separately (default: `['MMR', 'POS']`)

In [8]:
# =============================================================================
# REPRESENTATION MODEL - 3 LLM Provider Options
# =============================================================================

def create_llm_representation():
    """Create LLM representation model based on LLM_PROVIDER config.
    
    Note: BERTopic's LiteLLM class only supports these parameters:
    - model, prompt, nr_docs, diversity, delay_in_seconds, 
    - exponential_backoff, generator_kwargs
    
    doc_length and tokenizer are NOT supported by LiteLLM.
    """
    
    if LLM_PROVIDER == 'huggingface_api':
        # HuggingFace Inference API via LiteLLM
        return LiteLLM(
            model=LLM_MODEL,
            prompt=LLM_PROMPT,
            nr_docs=LLM_NR_DOCS,
            diversity=LLM_DIVERSITY,
            delay_in_seconds=LLM_DELAY,
            generator_kwargs={
                "max_tokens": 16,
                "temperature": 0.0,
                "stop": ["\n"]
            }
        )
    
    elif LLM_PROVIDER == 'local':
        # Local HuggingFace Transformers model
        print(f"Loading local LLM: {LLM_MODEL}")
        generator = hf_pipeline(
            'text-generation', 
            model=LLM_MODEL,
            device_map="auto",
            torch_dtype="auto"
        )
        return TextGeneration(
            generator,
            prompt=LLM_PROMPT,
            pipeline_kwargs={
                "do_sample": False,
                "max_new_tokens": 16,
                "num_return_sequences": 1,
            },
        )
    
    elif LLM_PROVIDER == 'openrouter':
        # OpenRouter via LiteLLM (OpenAI-compatible route)
        _llm_base = LLM_MODEL.split('openrouter/', 1)[1] if LLM_MODEL.startswith('openrouter/') else LLM_MODEL
        _llm_model = _llm_base if _llm_base.startswith('openai/') else f'openai/{_llm_base}'
        return LiteLLM(
            model=_llm_model,
            prompt=LLM_PROMPT,
            nr_docs=LLM_NR_DOCS,
            diversity=LLM_DIVERSITY,
            delay_in_seconds=LLM_DELAY,
            generator_kwargs={
                "api_base": OPENROUTER_API_BASE,
                "extra_body": {
                    "provider": OPENROUTER_PROVIDER_ROUTING
                },
                "max_tokens": 16,
                "temperature": 0.0,
                "stop": ["\n"],
            },
        )
    else:
        raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER}")


def create_representation_model():
    """Build representation model as Dictionary with optional Pipeline.
    
    Structure:
    - "Main": [pipeline_models..., LLM] - Sequential pipeline, result stored under "Main"
    - Additional keys: Parallel models stored separately for comparison
    
    This ensures all results are accessible via topic_model.topic_aspects_
    """
    
    # 1. Build pipeline models (run BEFORE LLM, sequentially)
    pipeline = []
    pipeline_info = []
    
    for model_name in PIPELINE_MODELS:
        if model_name == 'POS':
            pipeline.append(PartOfSpeech("en_core_web_sm"))
            pipeline_info.append("POS")
        elif model_name == 'KeyBERT':
            pipeline.append(KeyBERTInspired())
            pipeline_info.append("KeyBERT")
        elif model_name == 'MMR':
            pipeline.append(MaximalMarginalRelevance(diversity=MMR_DIVERSITY))
            pipeline_info.append(f"MMR(diversity={MMR_DIVERSITY})")
        else:
            raise ValueError(f"Unknown pipeline model: {model_name}")
    
    # 2. Add LLM at the end of pipeline
    pipeline.append(create_llm_representation())
    pipeline_info.append("LLM")
    
    # 3. Build result dictionary
    # If pipeline has multiple steps, use list; otherwise just the LLM
    result = {"Main": pipeline if len(pipeline) > 1 else pipeline[0]}
    
    # 4. Add parallel models (stored SEPARATELY in topic_aspects_)
    if 'MMR' in PARALLEL_MODELS:
        result["MMR"] = MaximalMarginalRelevance(diversity=MMR_DIVERSITY)
    
    if 'POS' in PARALLEL_MODELS:
        result["POS"] = PartOfSpeech("en_core_web_sm")
    
    if 'KeyBERT' in PARALLEL_MODELS:
        result["KeyBERT"] = KeyBERTInspired()
    
    # Print configuration summary
    print(f"Pipeline (Main): {' → '.join(pipeline_info)}")
    print(f"Parallel models: {[k for k in result.keys() if k != 'Main']}")
    print(f"Result keys: {list(result.keys())}")
    
    return result


# Build models
# Dynamic min_df: Use min_df=1 for small samples (few topics), min_df=2 for full data
_min_df = 1 if (SAMPLE_SIZE is not None and SAMPLE_SIZE < 1000) else 2
vectorizer_model = CountVectorizer(stop_words="english", min_df=_min_df, ngram_range=(1, 3))
print(f"CountVectorizer: min_df={_min_df}")

ctfidf_model = ClassTfidfTransformer()
representation_model = create_representation_model()

# embedding_model is needed for KeyBERTInspired and find_topics()
# KeyBERT requires a LOCAL embedding model even when using API for documents
if 'KeyBERT' in PIPELINE_MODELS or 'KeyBERT' in PARALLEL_MODELS:
    embedding_model = SentenceTransformer(KEYBERT_EMBEDDING_MODEL)
    print(f"KeyBERT embedding model: {KEYBERT_EMBEDDING_MODEL}")
elif EMBEDDING_PROVIDER == 'local':
    embedding_model = SentenceTransformer(EMBEDDING_MODEL)
else:
    # For API-based providers without KeyBERT: not needed
    embedding_model = None

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=BaseDimensionalityReduction(),  # Skip - we pass pre-computed embeddings
    hdbscan_model=clustering_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    top_n_words=20,
    verbose=True
)

# Fit model with pre-computed reduced embeddings
print(f"\nFitting BERTopic with LLM_PROVIDER={LLM_PROVIDER}...")
topics, probs = topic_model.fit_transform(documents, reduced_5d)

CountVectorizer: min_df=1
Loading local LLM: google/gemma-3-1b-it


Device set to use cpu


Pipeline (Main): POS → KeyBERT → MMR(diversity=0.3) → LLM
Parallel models: ['MMR', 'POS', 'KeyBERT']
Result keys: ['Main', 'MMR', 'POS', 'KeyBERT']


2025-12-09 11:23:58,193 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-12-09 11:23:58,194 - BERTopic - Dimensionality - Completed ✓
2025-12-09 11:23:58,195 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-12-09 11:23:58,200 - BERTopic - Cluster - Completed ✓
2025-12-09 11:23:58,204 - BERTopic - Representation - Fine-tuning topics using representation models.


KeyBERT embedding model: sentence-transformers/all-MiniLM-L6-v2

Fitting BERTopic with LLM_PROVIDER=local...


100%|██████████| 3/3 [07:47<00:00, 155.92s/it]
2025-12-09 11:31:53,302 - BERTopic - Representation - Completed ✓


## Outlier Reduction

In [9]:
# =============================================================================
# OUTLIER REDUCTION
# =============================================================================
# Ensure topics is a numpy array
topics = np.array(topic_model.topics_)

print(f"\n{'='*60}")
print("Outlier Reduction")
print(f"{'='*60}")
print(f"Before: {(topics == -1).sum():,} outliers ({(topics == -1).sum()/len(topics)*100:.1f}%)")

# Strategy: Embeddings - must have same dimensionality as fit_transform (5D)
new_topics = topic_model.reduce_outliers(
    documents, 
    topics, 
    strategy="embeddings",
    embeddings=reduced_5d,  # Must be reduced_5d, not embeddings (3072D)!
    threshold=0.8  # Adjustable: lower = more aggressive
)

# Update topic representations with new assignments
topic_model.update_topics(
    documents,
    topics=new_topics,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
)
topics = np.array(new_topics)

print(f"After:  {(topics == -1).sum():,} outliers ({(topics == -1).sum()/len(topics)*100:.1f}%)")

2025-12-09 11:31:53,433 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.



Outlier Reduction
Before: 1 outliers (0.5%)


100%|██████████| 3/3 [06:46<00:00, 135.55s/it]


After:  1 outliers (0.5%)


In [10]:
# =============================================================================
# EXTRACT LLM LABELS FROM MAIN REPRESENTATION
# =============================================================================
# Note: When using {"Main": [MMR, LLM], ...}, the "Main" pipeline result goes to
# topic_representations_ (the primary representation), NOT topic_aspects_.
# Only the parallel models (MMR, POS, KeyBERT) go to topic_aspects_.

print(f"Available topic_aspects_ keys: {list(topic_model.topic_aspects_.keys())}")
print(f"Number of topics in topic_representations_: {len(topic_model.topic_representations_)}")

# The LLM labels are in topic_representations_ (the main representation)
# Format: {topic_id: [(word1, score1), (word2, score2), ...]}
# For LLM output, word1 is typically the generated label

if topic_model.topic_representations_:
    llm_labels = {}
    for topic_id, representation in topic_model.topic_representations_.items():
        if representation:
            # LLM output: first element contains the label
            # Format varies: could be single label or multiple keywords
            words = [word for word, score in representation[:3]]  # Top 3
            llm_labels[topic_id] = " | ".join(words)
    
    llm_labels[-1] = "Outlier Topic"
    topic_model.set_topic_labels(llm_labels)
    print(f"✓ Labels set from topic_representations_ ({len(llm_labels)} topics)")
else:
    print("⚠ WARNING: topic_representations_ is empty.")
    print("  Using default c-TF-IDF labels instead.")

# Display topic info with all available aspects
topic_model.get_topic_info().head(10)

Available topic_aspects_ keys: ['MMR', 'POS', 'KeyBERT']
Number of topics in topic_representations_: 3
✓ Labels set from topic_representations_ (3 topics)


,Topic,Count,Name,CustomName,Representation,MMR,POS,KeyBERT,Representative_Docs
0,-1,1,"-1_\n```\n- Gagarin, Yuri Alekseyevich (1934___",Outlier Topic,"[\n```\n- Gagarin, Yuri Alekseyevich (1934, , ...","[gagarin, 1934 1968 russian, air force 1961, 1...","[man, son, cosmonaut, favourite son, favourite...","[gagarin yuri alekseyevich, russian cosmonaut ...","[Gagarin, Yuri Alekseyevich (1934-1968). Russi..."
1,0,164,0_\n```\ntopic: Galaxy Formation from Rotating...,\n```\ntopic: Galaxy Formation from Rotating G...,[\n```\ntopic: Galaxy Formation from Rotating ...,"[galaxies, gravitational, models, observed, eq...","[gravity, model, galaxies, theory, results, da...","[galaxies, cosmological, galactic, dark matter...",[Spectrophotometry of Galactic Supernova Remna...
2,1,35,1_\n```\n- Multicell Squall-Line Structure as ...,\n```\n- Multicell Squall-Line Structure as a ...,[\n```\n- Multicell Squall-Line Structure as a...,"[cosmology, gravity, cosmogenic, gravitons, al...","[cosmology, gravitation, ozone, alpine, gravit...","[vertically trapped gravity, trapped gravity w...",[Gravity anomalies of irregularly shaped two-d...


In [11]:
# =============================================================================
# ADD RESULTS TO DATAFRAME
# =============================================================================
# Add 2D coordinates for visualization (column names kept as UMAP-1/2 for compatibility)
df['UMAP-1'] = reduced_2d[:, 0]
df['UMAP-2'] = reduced_2d[:, 1]
df['Cluster'] = topics

# Map topic info columns to DataFrame
topic_info = topic_model.get_topic_info()

# Expected columns from our Dictionary representation model:
# - 'Main': LLM labels (pipeline result)
# - 'MMR': MMR-diversified keywords (if in PARALLEL_MODELS)
# - 'POS': Part-of-speech filtered keywords (if in PARALLEL_MODELS)
# - 'KeyBERT': KeyBERT re-ranked keywords (if in PARALLEL_MODELS)
expected_cols = ['Main', 'MMR', 'POS', 'KeyBERT']
base_cols = ['Name']

# Find which representation columns actually exist in topic_info
representation_cols = [col for col in expected_cols if col in topic_info.columns]
available_cols = base_cols + representation_cols

print(f"Available representation columns: {representation_cols}")
print(f"(From topic_aspects_ keys: {list(topic_model.topic_aspects_.keys())})")

for col in available_cols:
    if col in topic_info.columns:
        mapping = {
            row['Topic']: ', '.join(row[col]) if isinstance(row[col], list) else row[col] 
            for _, row in topic_info.iterrows()
        }
        df[col] = df['Cluster'].map(mapping)

df['full_embeddings'] = list(embeddings)

print(f"\nDataFrame shape: {df.shape}")
print(f"New columns: UMAP-1, UMAP-2, Cluster, {', '.join(available_cols)}, full_embeddings")

Available representation columns: ['MMR', 'POS', 'KeyBERT']
(From topic_aspects_ keys: ['MMR', 'POS', 'KeyBERT'])

DataFrame shape: (200, 32)
New columns: UMAP-1, UMAP-2, Cluster, Name, MMR, POS, KeyBERT, full_embeddings


In [12]:
df['References'] = df['References'].apply(lambda x: x if isinstance(x, list) else [])

df.to_parquet(OUTPUT_PARQUET, compression='snappy')
print(f"Saved DataFrame: {OUTPUT_PARQUET.name}")

topic_model.save(
    path=str(OUTPUT_MODEL),
    save_embedding_model=False,
    serialization="safetensors",
    save_ctfidf=True
)
print(f"Saved BERTopic model: {OUTPUT_MODEL.name}/")

2025-12-09 11:38:46,629 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you are using a sentence-transformers model or a HuggingFace model supportedby sentence-transformers, please save the model by using a pointer towards that model.For example, `save_embedding_model='sentence-transformers/all-mpnet-base-v2'`


Saved DataFrame: df_sampled_full_local_google_embeddinggemma-300m_pacmap_fast_hdbscan.parquet
Saved BERTopic model: bertopic_model_full_local_google_embeddinggemma-300m_pacmap_fast_hdbscan/


In [13]:
# =============================================================================
# COST SUMMARY
# =============================================================================
if TRACK_COSTS:
    print(f"\n{'='*60}")
    print("COST SUMMARY")
    print(f"{'='*60}")
    print(f"Embedding Provider:  {EMBEDDING_PROVIDER}")
    print(f"Embedding Model:     {EMBEDDING_MODEL}")
    print(f"LLM Provider:        {LLM_PROVIDER}")
    print(f"LLM Model:           {LLM_MODEL}")
    print(f"{'-'*60}")

    print(f"Embedding:           {total_embedding_tokens:,} tokens")
    print(f"Embedding Cost:      ${total_embedding_cost:.6f}")

    print(f"LLM Input:           {total_llm_input_tokens:,} tokens")
    print(f"LLM Output:          {total_llm_output_tokens:,} tokens")
    print(f"LLM Cost:            ${total_llm_cost:.6f}")

    print(f"{'-'*60}")
    total_cost = total_embedding_cost + total_llm_cost
    print(f"TOTAL ESTIMATED:     ${total_cost:.6f}")

if total_embedding_tokens == 0 and total_embedding_cost == 0:
    print("\n⚠ No embedding usage tracked. Embeddings may have been loaded from cache or the provider did not return usage.")

if total_llm_input_tokens == 0 and total_llm_output_tokens == 0:
    print("\n⚠ No LLM usage tracked. BERTopic may have skipped LLM calls or the provider did not return usage.")

if EMBEDDING_PROVIDER == 'openrouter' or LLM_PROVIDER == 'openrouter':
    print(f"\n📊 Verify actual costs: https://openrouter.ai/activity")

    print(f"{'='*60}")


COST SUMMARY
Embedding Provider:  local
Embedding Model:     google/embeddinggemma-300m
LLM Provider:        local
LLM Model:           google/gemma-3-1b-it
------------------------------------------------------------
Embedding:           0 tokens
Embedding Cost:      $0.000000
LLM Input:           0 tokens
LLM Output:          0 tokens
LLM Cost:            $0.000000
------------------------------------------------------------
TOTAL ESTIMATED:     $0.000000

⚠ No embedding usage tracked. Embeddings may have been loaded from cache or the provider did not return usage.

⚠ No LLM usage tracked. BERTopic may have skipped LLM calls or the provider did not return usage.
